# 05.02 · Chat APIs deep dive

**Chapter:** Working with LLM APIs · **Milestone:** M2

## Goals
- [ ] Messages and roles; system prompts; multi-turn state (the API is stateless — *you* keep history)
- [ ] Key parameters: `temperature`, `max_tokens`, `stop`, `seed`; `finish_reason`
- [ ] Streaming
- [ ] Same call through three routes: OpenAI-compatible endpoint, native SDKs (Anthropic, Google), and LiteLLM

## Resources
- [OpenAI API docs](https://platform.openai.com/docs) · [Anthropic API docs](https://docs.anthropic.com/) · [Gemini API docs](https://ai.google.dev/gemini-api/docs) — free
- [LiteLLM docs](https://docs.litellm.ai/) — free
- [Anthropic courses (API fundamentals)](https://github.com/anthropics/courses) — free

| Route | When |
|---|---|
| **OpenAI-compatible endpoint** (`openai` SDK + `base_url`) | most providers & local servers speak it; simplest multi-provider code |
| **Native SDK** (`anthropic`, `google-genai`, `openai`) | provider-specific features first (caching controls, thinking, files, citations) |
| **LiteLLM** / aisuite | one function, 100+ providers, unified errors/costs; also a proxy server (chapter 11) |

In [ ]:
from aieng.llm import available, chat, client

c, MODEL = client()  # first available provider; set AIENG_PROVIDER in .env to choose
LOCAL = c.base_url.host == "localhost"
EXTRA = {"reasoning_effort": "none"} if LOCAL else {}  # disable slow "thinking" on local qwen3
print("using", c.base_url, MODEL)

## 1. Messages and roles
- `system`: instructions, persona, rules. Highest-priority guidance.
- `user`: the human's turns.
- `assistant`: the model's previous turns (you send them back for multi-turn).
- `tool`: results of tool calls (chapter 09).

In [ ]:
resp = c.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a senior Python reviewer. Reply in at most 2 sentences."},
        {"role": "user", "content": "Is `def f(x=[]):` a problem?"},
    ],
    **EXTRA,
)
print(resp.choices[0].message.content)
print("finish_reason:", resp.choices[0].finish_reason, "| usage:", resp.usage.prompt_tokens, "in /", resp.usage.completion_tokens, "out")

## 2. The API is stateless: you own the history
Every request must contain the whole conversation. That's why long chats get slower and pricier: the input grows every turn.

In [ ]:
class Chat:
    def __init__(self, system, provider=None):
        self.client, self.model = client(provider)
        self.messages = [{"role": "system", "content": system}]

    def send(self, text, **kwargs):
        self.messages.append({"role": "user", "content": text})
        r = self.client.chat.completions.create(model=self.model, messages=self.messages, **{**EXTRA, **kwargs})
        reply = r.choices[0].message.content
        self.messages.append({"role": "assistant", "content": reply})
        print(f"[input tokens this turn: {r.usage.prompt_tokens}]")
        return reply


bot = Chat("You are a helpful assistant. Be brief.")
print(bot.send("My name is Ana and I'm learning AI engineering."))
print(bot.send("What's my name, and what am I learning?"))

## 3. Parameters
- `temperature` (0–2): randomness (04.06). ~0 for extraction/classification, 0.7–1 for creative.
- `max_tokens` / `max_completion_tokens`: hard cap on output → `finish_reason="length"` if hit. Always check it.
- `stop`: strings that end generation early.
- `seed`: best-effort reproducibility.
- `n`: several completions in one call (not supported everywhere).

In [ ]:
for kwargs in [{"max_tokens": 15}, {"stop": ["3"]}, {"temperature": 0}]:
    r = c.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "Count from 1 to 10, comma-separated."}], **EXTRA, **kwargs)
    print(f"{str(kwargs):22} finish={r.choices[0].finish_reason:7} -> {r.choices[0].message.content!r}")

## 4. Streaming
`stream=True` returns an iterator of chunks (the generator idea from 01.01). Users see text immediately; essential for chat UIs.

In [ ]:
stream = c.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "Give 3 tips for writing clean Python."}],
                                   stream=True, **EXTRA)
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()

## 5. Native SDKs
Same request, different shapes. Note Anthropic's differences: `system` is a top-level parameter, `max_tokens` is required,
and the response is a list of content blocks. These cells run only if you have the keys.

In [ ]:
import os

if os.getenv("ANTHROPIC_API_KEY"):
    import anthropic
    a = anthropic.Anthropic()
    msg = a.messages.create(model="claude-haiku-4-5", max_tokens=200, system="Be brief.",
                            messages=[{"role": "user", "content": "What is a vector database?"}])
    print("Anthropic:", msg.content[0].text, "| usage:", msg.usage.input_tokens, msg.usage.output_tokens)
else:
    print("ANTHROPIC_API_KEY not set (paid) — skipping")

if os.getenv("GEMINI_API_KEY"):
    from google import genai
    g = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    r = g.models.generate_content(model="gemini-flash-latest", contents="What is a vector database? Be brief.")
    print("Gemini native:", r.text)
else:
    print("GEMINI_API_KEY not set (free at aistudio.google.com) — skipping")

## 6. LiteLLM: one function for every provider
Model strings are `provider/model`. Swap providers by changing a string; errors and usage are normalised.

In [ ]:
import litellm

litellm.suppress_debug_info = True
targets = ["ollama_chat/qwen3:1.7b"]
if os.getenv("GEMINI_API_KEY"): targets.append("gemini/gemini-flash-latest")
if os.getenv("GROQ_API_KEY"): targets.append("groq/llama-3.3-70b-versatile")

for target in targets:
    kwargs = {"api_base": "http://localhost:11434", "reasoning_effort": "none"} if target.startswith("ollama") else {}
    try:
        r = litellm.completion(model=target, messages=[{"role": "user", "content": "One-sentence definition of an embedding."}], **kwargs)
        print(f"{target}: {r.choices[0].message.content}")
    except Exception as e:
        print(f"{target}: {type(e).__name__}: {str(e)[:100]}")

## Exercises — toward Milestone M2
1. Turn `Chat` into a **streaming terminal chat**: a script `chat.py` with a `while True: input()` loop, streamed replies, `/reset`
   and `/history` commands, provider chosen by `AIENG_PROVIDER`. (Run it in a terminal, not the notebook.)
2. Add a token budget: when the history exceeds N tokens, drop the oldest user/assistant pairs (keep the system message).
3. Write the same 3-turn conversation with the native Anthropic or Gemini SDK. List every difference from the OpenAI format.

In [ ]:
# your code here